# Text Editor Undo-Redo menggunakan Stack dan Doubly Linked List

Struktur sel:

1. Impor & objek `Action`
2. `ActionStack` (doubly linked list)
3. `TextEditor`
4. Pengujian
5. Tampilan (CSS & helper HTML)
6. Kelas aplikasi `UndoRedoApp`
7. Jalankan aplikasi

## 1. Impor & objek Action

In [8]:
import gc
import random
import weakref
from dataclasses import dataclass
from html import escape

import ipywidgets as widgets
from IPython.display import display

INSERT = "INSERT"
DELETE = "DELETE"


# ---------------------------------------------------------------------
#  Objek Action: satu perubahan pada dokumen (bukan snapshot string)
# ---------------------------------------------------------------------
@dataclass
class Action:
    type: str            # INSERT / DELETE
    pos: int             # letak perubahan
    text: str            # teks yang disisipkan / dihapus
    cursor_before: int   # kursor sebelum aksi
    cursor_after: int    # kursor sesudah aksi

## 2. ActionStack (Doubly Linked List)

In [9]:
# ---------------------------------------------------------------------
#  ActionStack: stack berbasis doubly linked list
#     top    -> aksi terbaru (push/pop terjadi di sini)
#     bottom -> aksi terlama (dibuang bila riwayat melebihi batas)
# ---------------------------------------------------------------------
class _Node:
    __slots__ = ("data", "below", "above", "__weakref__")

    def __init__(self, data, below):
        self.data = data
        self.below = below   # menuju elemen yang lebih lama
        self.above = None    # menuju elemen yang lebih baru


class ActionStack:
    def __init__(self):
        self._top = None
        self._bottom = None
        self._size = 0

    def __len__(self):
        return self._size

    def is_empty(self):
        return self._size == 0

    def push(self, action):                      # O(1)
        node = _Node(action, self._top)
        if self._top is not None:
            self._top.above = node
        else:
            self._bottom = node                  # stack kosong
        self._top = node
        self._size += 1

    def pop(self):                               # O(1)
        """Keluarkan aksi teratas. Mengembalikan None bila kosong (underflow aman)."""
        node = self._top
        if node is None:
            return None
        self._top = node.below
        if self._top is not None:
            self._top.above = None
        else:
            self._bottom = None
        data = node.data
        node.below = node.above = node.data = None   # putus semua referensi
        self._size -= 1
        return data

    def peek(self):                              # O(1)
        return self._top.data if self._top is not None else None

    def drop_oldest(self):                       # O(1)
        """Buang aksi terlama saat riwayat melebihi batas."""
        node = self._bottom
        if node is None:
            return
        self._bottom = node.above
        if self._bottom is not None:
            self._bottom.below = None
        else:
            self._top = None
        node.below = node.above = node.data = None
        self._size -= 1

    def clear(self):                             # O(n)
        while self._top is not None:
            self.pop()

    def __iter__(self):
        """Iterasi dari top (terbaru) ke bottom (terlama)."""
        node = self._top
        while node is not None:
            yield node.data
            node = node.below

## 3. TextEditor (dokumen + kursor + Undo/Redo Stack)

In [10]:
# ---------------------------------------------------------------------
#  TextEditor: dokumen + kursor + Undo Stack + Redo Stack
# ---------------------------------------------------------------------
class TextEditor:
    def __init__(self, max_history=100):
        self.text = ""
        self.cursor = 0
        self.max_history = max_history
        self._undo = ActionStack()
        self._redo = ActionStack()

    # ---- properti baca ----
    @property
    def undo_stack(self):
        return self._undo

    @property
    def redo_stack(self):
        return self._redo

    # ---- pencatatan ----
    def _record(self, action):
        self._redo.clear()               # aksi baru membatalkan jalur "masa depan"
        self._undo.push(action)
        if len(self._undo) > self.max_history:
            self._undo.drop_oldest()

    # ---- operasi penyuntingan ----
    def move_cursor(self, p):
        self.cursor = max(0, min(int(p), len(self.text)))

    def insert(self, s):
        """Sisipkan teks di posisi kursor. Mengembalikan Action (atau None)."""
        if not s:
            return None
        c = self.cursor
        self.text = self.text[:c] + s + self.text[c:]
        self.cursor = c + len(s)
        action = Action(INSERT, c, s, c, self.cursor)
        self._record(action)
        return action

    def delete_range(self, start, length):
        """Hapus 'length' karakter mulai dari 'start' (seperti menghapus seleksi)."""
        if start < 0 or start >= len(self.text) or length <= 0:
            return None
        end = min(len(self.text), start + length)
        removed = self.text[start:end]
        before = self.cursor
        self.text = self.text[:start] + self.text[end:]
        self.cursor = start
        action = Action(DELETE, start, removed, before, start)
        self._record(action)
        return action

    def backspace(self, n=1):
        """Hapus n karakter tepat sebelum kursor sebagai SATU Action."""
        start = max(0, self.cursor - max(1, int(n)))
        return self.delete_range(start, self.cursor - start)

    # ---- undo / redo ----
    def undo(self):
        a = self._undo.pop()
        if a is None:
            return None                  # tidak ada yang bisa di-undo
        if a.type == INSERT:
            self.text = self.text[:a.pos] + self.text[a.pos + len(a.text):]
        else:
            self.text = self.text[:a.pos] + a.text + self.text[a.pos:]
        self.cursor = a.cursor_before
        self._redo.push(a)
        return a

    def redo(self):
        a = self._redo.pop()
        if a is None:
            return None                  # tidak ada yang bisa di-redo
        if a.type == INSERT:
            self.text = self.text[:a.pos] + a.text + self.text[a.pos:]
        else:
            self.text = self.text[:a.pos] + self.text[a.pos + len(a.text):]
        self.cursor = a.cursor_after
        self._undo.push(a)
        if len(self._undo) > self.max_history:
            self._undo.drop_oldest()
        return a

## 4. Pengujian (push/pop, undo/redo, batas riwayat, stress, memory leak)

In [11]:
# =====================================================================
#  PENGUJIAN (kebenaran push/pop, tanpa memory leak / error)
# =====================================================================
def test_basic():
    ed = TextEditor()
    assert ed.undo() is None and ed.redo() is None       # underflow aman

    ed.insert("abc"); ed.insert("\n"); ed.insert("def")
    assert ed.text == "abc\ndef" and ed.cursor == 7
    assert len(ed.undo_stack) == 3

    ed.undo()
    assert ed.text == "abc\n" and ed.cursor == 4
    ed.undo()
    assert ed.text == "abc" and ed.cursor == 3
    ed.redo()
    assert ed.text == "abc\n" and ed.cursor == 4

    ed.move_cursor(3)                                    # hapus rentang + kursor
    ed.delete_range(0, 3)
    assert ed.text == "\n" and ed.cursor == 0
    ed.undo()
    assert ed.text == "abc\n" and ed.cursor == 3         # kursor sebelum aksi

    ed.insert("X")                                       # aksi baru -> redo bersih
    assert len(ed.redo_stack) == 0 and ed.redo() is None


def test_history_limit():
    ed = TextEditor(max_history=5)
    for _ in range(20):
        ed.insert("\n")
    assert len(ed.undo_stack) == 5
    n = 0
    while ed.undo():
        n += 1
    assert n == 5 and len(ed.text) == 15


def test_stress(seed, ops=20000):
    rng = random.Random(seed)
    ed = TextEditor(max_history=10 ** 9)
    for _ in range(ops):
        k = rng.randrange(6)
        if k <= 1:
            ed.insert("".join(rng.choice("abcdefghij") for _ in range(rng.randint(1, 3))))
        elif k == 2:
            ed.backspace(rng.randint(1, 3))
        elif k == 3:
            ed.delete_range(ed.cursor, 1)
        elif k == 4:
            ed.delete_range(rng.randint(0, len(ed.text)), rng.randint(1, 4))
        else:
            ed.move_cursor(rng.randint(0, len(ed.text)))
    ed.insert("Z")
    final_text, final_cursor = ed.text, ed.cursor

    while ed.undo():
        pass
    assert ed.text == "" and ed.cursor == 0 and len(ed.undo_stack) == 0

    while ed.redo():
        pass
    assert ed.text == final_text and ed.cursor == final_cursor


def test_no_leak():
    """Semua node harus langsung bebas (tanpa bantuan GC) setelah dilepas."""
    gc.disable()
    try:
        st = ActionStack()
        refs = []
        for i in range(5000):
            st.push(Action(INSERT, i, "x" * 20, i, i + 20))
            refs.append(weakref.ref(st._top))
        for _ in range(1000):
            st.pop()
        for _ in range(1000):
            st.drop_oldest()
        st.clear()
        assert len(st) == 0 and st._top is None and st._bottom is None
        assert all(r() is None for r in refs), "ada node yang bocor"
    finally:
        gc.enable()


def run_tests():
    test_basic()
    test_history_limit()
    for s in range(1, 4):
        test_stress(s)
    test_no_leak()
    print("✔ Self-test lulus: push/pop, undo/redo, batas riwayat, stress 3x20.000 operasi, tanpa leak.")

## 5. Tampilan: CSS & helper HTML

In [12]:
MONO = "'Consolas','Courier New',monospace"

CSS = """
<style>
.case-root { font-family: Arial, sans-serif; color: #111 !important; }
.case-root .widget-button button {
    border-radius: 5px !important;
    font-weight: 700 !important;
    min-height: 38px !important;
    box-shadow: none !important;
}
.case-primary button {
    background: #111 !important; color: #fff !important;
    border: 1px solid #111 !important;
}
.case-outline button {
    background: #fff !important; color: #111 !important;
    border: 1px solid #111 !important;
}
.case-root input {
    border: 1px solid #aaa !important;
    border-radius: 5px !important;
    min-height: 32px !important;
    background: #fff !important;
    color: #111 !important;
}
.case-root label { color: #333 !important; font-weight: 600 !important; }
</style>
"""


def _shorten(t, n=24):
    t = t.replace("\\n", "↵")
    return t if len(t) <= n else t[: n - 1] + "…"


def _header_html():
    return """
    <div style="background:#111;color:#fff;padding:20px 22px;border-radius:7px;
                font-family:Arial,sans-serif">
      <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap">
        <span style="background:#fff;color:#111;padding:7px 11px;border-radius:4px;
                     font-size:12px;font-weight:800;letter-spacing:1px">CASE METHOD</span>
        <span style="font-size:27px;font-weight:800;letter-spacing:.2px">
          Undo-Redo Editor
        </span>
      </div>
      <div style="color:#ddd;font-size:14px;line-height:1.6;margin-top:12px">
        Simulasi dua Stack (Undo dan Redo) pada aplikasi teks editor.
        Setiap perubahan dicatat sebagai aksi agar dapat dibatalkan atau diulang.
      </div>
    </div>
    """


def _doc_html(text, cursor):
    caret = '<span style="display:inline-block;width:2px;height:20px;background:#111;vertical-align:middle;margin:0 2px"></span>'
    if text:
        body = escape(text[:cursor]) + caret + escape(text[cursor:])
    else:
        body = caret + '<span style="color:#999">(dokumen kosong)</span>'
    return f"""
    <div style="background:#fff;border:1px solid #bbb;border-left:5px solid #111;
                border-radius:4px;padding:18px;color:#111;font-family:{MONO};
                font-size:17px;line-height:1.7;white-space:pre-wrap;
                word-break:break-word;min-height:88px">{body}</div>
    """


def _stack_html(title, stack):
    rows = []
    for i, action in enumerate(stack):
        top = "  • TOP" if i == 0 else ""
        rows.append(f"""
        <div style="border:1px solid #aaa;background:#fff;border-radius:4px;
                    padding:9px 10px;margin-bottom:6px;color:#111">
          <div style="font-size:12px;font-weight:800;letter-spacing:.5px">
            {action.type}{top}
          </div>
          <div style="font-family:{MONO};font-size:13px;margin-top:5px;
                      overflow-wrap:anywhere">
            "{escape(_shorten(action.text, 30))}"
          </div>
          <div style="font-size:10px;color:#555;margin-top:5px">
            Posisi {action.pos} · Kursor {action.cursor_before} → {action.cursor_after}
          </div>
        </div>
        """)
    body = "".join(rows) if rows else (
        '<div style="color:#777;text-align:center;padding:28px 5px;font-size:13px">'
        'Belum ada aksi</div>'
    )
    return f"""
    <div style="background:#f5f5f5;border:1px solid #888;border-radius:5px;
                padding:12px;width:100%;height:310px;box-sizing:border-box;
                display:flex;flex-direction:column">
      <div style="font-family:{MONO};font-size:15px;text-align:center;
                  color:#111;padding:4px 0 12px;border-bottom:1px solid #aaa;
                  margin-bottom:10px">
        <b>{title}</b> ({len(stack)})
      </div>
      <div style="overflow-y:auto;flex:1">{body}</div>
    </div>
    """

## 6. Kelas aplikasi UndoRedoApp (ipywidgets)

In [13]:
class UndoRedoApp:
    def __init__(self, max_history=50):
        self.ed = TextEditor(max_history=max_history)
        self._syncing = False

        self.css = widgets.HTML(CSS)
        self.header = widgets.HTML(_header_html())

        self.txt = widgets.Text(
            placeholder="Ketik kalimat di sini...",
            layout=widgets.Layout(width="100%", height="42px")
        )

        self.btn_write = widgets.Button(
            description="Write / Save",
            layout=widgets.Layout(width="100%", height="40px")
        )
        self.btn_undo = widgets.Button(
            description="↶ Undo",
            layout=widgets.Layout(width="100%", height="40px")
        )
        self.btn_redo = widgets.Button(
            description="↷ Redo",
            layout=widgets.Layout(width="100%", height="40px")
        )
        self.btn_del = widgets.Button(
            description="Backspace",
            layout=widgets.Layout(width="100%", height="36px")
        )
        self.btn_write.add_class("case-primary")
        self.btn_undo.add_class("case-outline")
        self.btn_redo.add_class("case-outline")
        self.btn_del.add_class("case-outline")

        self.cursor_in = widgets.BoundedIntText(
            value=0, min=0, max=0, description="Kursor",
            layout=widgets.Layout(width="48%")
        )
        self.del_n = widgets.BoundedIntText(
            value=1, min=1, max=1000, description="Hapus",
            layout=widgets.Layout(width="48%")
        )

        self.label = widgets.HTML(
            '<div style="font-family:Consolas,monospace;font-size:13px;'
            'color:#555;margin:10px 0 7px">CURRENT DOCUMENT STATE</div>'
        )
        self.doc = widgets.HTML()
        self.status = widgets.HTML()
        self.undo_panel = widgets.HTML()
        self.redo_panel = widgets.HTML()

        self.btn_write.on_click(self._on_write)
        self.btn_undo.on_click(self._on_undo)
        self.btn_redo.on_click(self._on_redo)
        self.btn_del.on_click(self._on_delete)
        self.cursor_in.observe(self._on_cursor, names="value")

        left = widgets.VBox([
            self.txt,
            widgets.HBox(
                [self.btn_write, self.btn_undo, self.btn_redo],
                layout=widgets.Layout(gap="8px", width="100%")
            ),
            widgets.HBox(
                [self.cursor_in, self.del_n],
                layout=widgets.Layout(gap="8px", width="100%")
            ),
            self.btn_del,
            self.label,
            self.doc,
            self.status
        ], layout=widgets.Layout(
            width="52%", padding="4px 12px 10px 0", gap="9px"
        ))

        right = widgets.HBox(
            [self.undo_panel, self.redo_panel],
            layout=widgets.Layout(width="48%", gap="12px")
        )

        workspace = widgets.HBox(
            [left, right],
            layout=widgets.Layout(width="100%", align_items="stretch")
        )

        analysis_note = widgets.HTML("""
        <div style="border:1px solid #aaa;border-radius:5px;padding:12px 14px;
                    color:#222;background:#fff;font-family:Arial,sans-serif">
          <div style="font-weight:800;font-size:14px;margin-bottom:5px">
            Analisis Permasalahan & Arsitektur Data
          </div>
          <div style="font-size:12px;line-height:1.6;color:#444">
            Riwayat perubahan disimpan sebagai <b>Doubly Linked List dinamis</b>.
            Struktur ini dipilih karena jumlah aksi tidak tetap, operasi push/pop
            di TOP berkompleksitas O(1), dan node dapat dilepas tanpa menggeser
            elemen lain. Array statis memerlukan kapasitas tetap sejak awal.
          </div>
        </div>
        """)

        self.ui = widgets.VBox([
            self.css,
            self.header,
            workspace,
            analysis_note
        ], layout=widgets.Layout(
            width="100%", padding="14px", gap="14px",
            background_color="#fff"
        ))
        self.ui.add_class("case-root")
        self._refresh("Mulai dengan mengetik teks, lalu tekan Write / Save.")

    def _render_doc(self):
        self.doc.value = _doc_html(self.ed.text, self.ed.cursor)

    def _refresh(self, msg=None):
        self._syncing = True
        self.cursor_in.max = len(self.ed.text)
        self.cursor_in.value = self.ed.cursor
        self._syncing = False
        self._render_doc()
        self.undo_panel.value = _stack_html("UNDO STACK", self.ed.undo_stack)
        self.redo_panel.value = _stack_html("REDO STACK", self.ed.redo_stack)
        self.btn_undo.disabled = self.ed.undo_stack.is_empty()
        self.btn_redo.disabled = self.ed.redo_stack.is_empty()
        if msg is not None:
            self.status.value = (
                f'<div style="font-size:12px;color:#444;margin-top:5px">{msg}</div>'
            )

    def _on_write(self, _):
        t = self.txt.value
        if not t:
            self._refresh("Kolom teks masih kosong.")
            return
        a = self.ed.insert(t)
        self.txt.value = ""
        self._refresh(
            f'INSERT "{escape(_shorten(a.text, 40))}" pada posisi {a.pos}.'
        )

    def _on_delete(self, _):
        a = self.ed.backspace(self.del_n.value)
        if a is None:
            self._refresh("Tidak ada karakter di sebelah kiri kursor.")
            return
        self._refresh(
            f'DELETE "{escape(_shorten(a.text, 40))}" pada posisi {a.pos}.'
        )

    def _on_undo(self, _):
        a = self.ed.undo()
        if a is None:
            self._refresh("Undo Stack kosong.")
            return
        self._refresh(f'UNDO berhasil: "{escape(_shorten(a.text, 40))}".')

    def _on_redo(self, _):
        a = self.ed.redo()
        if a is None:
            self._refresh("Redo Stack kosong.")
            return
        self._refresh(f'REDO berhasil: "{escape(_shorten(a.text, 40))}".')

    def _on_cursor(self, change):
        if self._syncing:
            return
        self.ed.move_cursor(change["new"])
        self._render_doc()

## 7. Jalankan aplikasi

In [14]:
def main(tests=True):
    if tests:
        run_tests()
    app = UndoRedoApp()
    display(app.ui)
    return app


app = main()

✔ Self-test lulus: push/pop, undo/redo, batas riwayat, stress 3x20.000 operasi, tanpa leak.
